# DataOps: does a new field season change how body mass tracks flipper length in penguins?

You study body mass against flipper length in penguins. How much heavier is a penguin per mm of flipper, within a species? You have the 2007 field season, and the answer is about 40 g/mm. Then the field station sends 2008 and 2009. Your pipeline runs without a complaint, and the answer drops to under 1 g/mm.

Nothing crashed. The data is wrong in a way your pipeline doesn't look for. You'll make it look, roll back to the good result, repair the data on the record, and test that your check can fail. The data is a pet, so nobody edits it by hand. It has versions, and a repair is a recorded decision.

**The short path takes about 15 minutes:** do step 1, then run its check. Steps 2 and 3 are optional and take about 20 more. Only reading? Each step shows the failure and says how to approach the fix.

## Get a terminal

You read this page and paste commands into a terminal. You don't run anything on the page. The output shown here is from a real run of these same commands on the untouched project, so it stops at the failures. The fixes are yours.

**[Open it in Binder](https://mybinder.org/v2/gh/ohpcsim/reproducibility-in-ai/binder-02-dataops?urlpath=lab/workspaces/exercise).** It's free and there's nothing to install. The first start can take a few minutes. You get a terminal on the left, already in the `project` folder, and this page on the right. To copy a command, select it and press Ctrl+C, or Cmd+C on a Mac. To paste it in the terminal, press Ctrl+Shift+V, or Cmd+V on a Mac, or right-click and choose Paste. To edit a file, double-click it in the file panel on the far left, and save with Ctrl+S. A Binder session is temporary and has 2 GB of memory, so download anything you want to keep.

**Or use your laptop's terminal,** with this page open in your browser. It works on Linux and macOS, on x86_64 or ARM, and on Windows through WSL2. You need [conda](https://github.com/conda-forge/miniforge), and you don't need Jupyter. On Windows, keep the folder in your WSL2 home, not under `/mnt/c`. Paste this once:

```bash
git clone --branch binder-02-dataops --single-branch https://github.com/ohpcsim/reproducibility-in-ai.git dataops
cd dataops
conda env create --quiet --file .binder/environment.yml
conda activate repro-dataops
python scripts/setup_project.py
cd project
```

Every command below runs in that `project` folder. To start over, go to the folder above it and run `python scripts/setup_project.py --reset`, which deletes your work.

## The project

`project` already has three scripts, a cleaning policy, and the 2007 data under DVC. `../osn-bucket` stands in for an object storage bucket such as OSN. DVC uses a real bucket the same way, and your keys stay in `--local` config, out of Git.

These are the commands that set it up, so you can use them on your own project. `dvc add` puts a small pointer file in Git and keeps the data out of it. Each `dvc stage add` declares a step, what it reads, and what it writes. `dvc repro` runs whatever is out of date. `dvc push` sends the data to the bucket. `clean` comes first and `validate` checks its output, so `report` only ever reads data that passed. You don't need to run these.

```text
dvc init
dvc remote add -d storage ../osn-bucket
dvc add data/penguins.csv
dvc stage add -n clean -d data/penguins.csv -d clean.py -o results/clean.csv python clean.py
dvc stage add -n validate -d results/clean.csv -d validate.py -o results/validated.csv python validate.py
dvc stage add -n report -d results/validated.csv -d report.py -M metrics.json -o results/report.md python report.py
dvc repro
git add .
git commit -m "Track the 2007 season and the pipeline"
dvc push
```

Have a look at it.

```bash
git log --format='%h  %s'
cat params.yaml
dvc metrics show
```

In [1]:
# Recorded output. You don't need to run this.
import sys

sys.path.insert(0, ".exercise")
from page import run

run("git log --format='%h  %s'", "cat params.yaml", "dvc metrics show")

$ git log --format='%h  %s'
4af2375  Track the 2007 season and the pipeline
c745b64  Add the analysis scripts and cleaning policy

$ cat params.yaml
clean:
  # Drop penguins with no recorded sex.
  drop_missing_sex: false
  # Repairs for a bad delivery. They stay off until a person decides to turn one on.
  convert_inches: false
  drop_duplicates: false

$ dvc metrics show
Path          n_rows    slope_g_per_mm
metrics.json  109       40.659


## Step 1: a delivery that passes and shouldn't

The field station sends the rest. Take the delivery, update the pointer, and rerun, the way you would on a normal day.

```bash
python ../scripts/new_season.py
dvc add data/penguins.csv
dvc repro
dvc metrics diff
```

In [2]:
# Recorded output. You don't need to run this.
run("python ../scripts/new_season.py", "dvc add data/penguins.csv", "dvc repro", "dvc metrics diff")

$ python ../scripts/new_season.py
Delivered 356 rows to project/data/penguins.csv, covering 2007 to 2009.

$ dvc add data/penguins.csv

To track the changes with git, run:

	git add data/penguins.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true

$ dvc repro
'data/penguins.csv.dvc' didn't change, skipping
Running stage 'clean':
> python clean.py
clean: kept 354 rows
Updating lock file 'dvc.lock'

Running stage 'validate':
> python validate.py
validate: 354 rows passed
Updating lock file 'dvc.lock'

Running stage 'report':
> python report.py
| species | rows | slope (g/mm) |
|---|---|---|
| Adelie | 158 | 0.4 |
| Chinstrap | 69 | 1.3 |
| Gentoo | 127 | -0.0 |
| within species | 354 | 0.3 |

Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.

$ dvc metrics diff
Path          Metric          HEAD    workspace    Chang

The pipeline ran green, and the slope collapsed. A longer flipper can't mean almost no extra mass. Look at the delivery yourself. `validate.py` checks that the columns exist and that there are rows, which is all it was ever asked to do.

```bash
python ../scripts/look_at_delivery.py
```

In [3]:
# Recorded output. You don't need to run this.
run("python ../scripts/look_at_delivery.py")

$ python ../scripts/look_at_delivery.py
       flipper_length_mm  body_mass_g
count              354.0        354.0
mean               178.1       4205.2
std                 63.0        800.1
min                  7.3       2700.0
25%                187.0       3550.0
50%                195.0       4050.0
75%                210.0       4768.8
max                231.0       6300.0
repeated rows: 12


Your turn:

1. Add checks to `validate.py` until `dvc repro` stops at `validate` on this delivery. Decide what a believable flipper length is, and what counts as a repeated row. Exit with a message a tired colleague can act on, such as `sys.exit("validate: ...")`. The check below will plant more than one kind of bad delivery, including a kind this one doesn't have, so think about the `species` column too.
2. Run it and watch it stop. The metrics don't change, because `report` never ran.

```bash
dvc repro
```

3. Roll back. Git restores the pointer, `dvc checkout` restores the data, and `dvc metrics diff` should print nothing. That's how you know you're back. The rollback names the files, because a bare `git checkout .` would also throw away your new `validate.py`.

```bash
git checkout -- data/penguins.csv.dvc dvc.lock metrics.json
dvc checkout
dvc metrics diff
```

4. Rerun on the good data and commit your checks.

```bash
dvc repro
git add validate.py dvc.lock
git commit --quiet -m "Validate the range and uniqueness of a delivery"
dvc push
```

### Check the short path

Run this when step 1 is done. It clones your last commit into an empty folder, pulls the data from your bucket, and plants its own bad deliveries. Each line says what to do next. When it passes, a bad delivery stops your pipeline and you've finished the short path. You can stop here.

```bash
python ../check.py --short
```

With the bad delivery in place and no new checks, it fails like this:

In [4]:
# Recorded output. You don't need to run this.
run("python ../check.py --short")

$ python ../check.py --short
Note: you have uncommitted changes. This checks your last commit only.

PASS  The pipeline is committed
PASS  A fresh clone pulls everything into an empty cache
FAIL  dvc metrics diff in project/ isn't empty, so your workspace doesn't match your last commit. Roll back with the commands in step 1, or commit the result you want to keep.
PASS  Good data passes validate
FAIL  I planted a delivery with flipper lengths in inches, with the repairs off, and dvc repro did not stop at validate. Add a check to validate.py.
FAIL  I planted a delivery with repeated rows, with the repairs off, and dvc repro did not stop at validate. Add a check to validate.py.
FAIL  I planted a delivery with a misspelled species, with the repairs off, and dvc repro did not stop at validate. Add a check to validate.py.

3 of 7 parts pass.


## Step 2 (optional): a policy nobody declared

Refusing the delivery is right for now, but you still want those two seasons. Most of the rows are good. A person should decide how to repair the rest, and Git should record that decision. That's what the policy in `params.yaml` is for.

There's a problem with it, though. This script changes the policy, asks DVC what's out of date, and puts the policy back.

```bash
python ../scripts/try_policy.py
```

In [5]:
# Recorded output. You don't need to run this.
run("python ../scripts/try_policy.py")

$ python ../scripts/try_policy.py
I changed drop_missing_sex in params.yaml. A different rule means different rows. dvc status says:

Data and pipelines are up to date.

I put params.yaml back.


DVC doesn't see the change. `clean.py` reads `params.yaml`, but `dvc.yaml` doesn't say so. Change the policy and your report is stale, with `dvc repro` telling you everything is up to date. It's the same failure as exercise 1, a number that no longer matches its inputs.

Your turn:

1. Declare the `clean` policy as a dependency of the `clean` stage in `dvc.yaml`. A stage can have a `params:` list (the [DVC docs](https://doc.dvc.org/user-guide/project-structure/dvcyaml-files) show the format). Run `try_policy.py` again. It should now report the change.

```bash
python ../scripts/try_policy.py
```

2. In `clean.py`, act on `policy["convert_inches"]` and `policy["drop_duplicates"]`. Leave each repair off unless its setting is on. Don't hand-edit the delivery.
3. Turn both on in `params.yaml`. Your `validate.py` stays as strict as it was, and it now checks the repaired data.
4. Take the delivery again, rerun, and compare. You're looking for a slope close to the first season's.

```bash
python ../scripts/new_season.py
dvc add data/penguins.csv
dvc repro
dvc metrics diff
```

5. Commit and push. Git holds the policy and the pointer, and the bucket holds the data, so you need both.

```bash
git add .
git commit --quiet -m "Accept the 2008 and 2009 seasons, repaired by the recorded policy"
dvc push
```

## Step 3 (optional): test that the result can move

A pipeline that has never failed on known-wrong input hasn't been tested. Your slope matched, but would it have noticed a change? Change one row of the data, in a way `validate` still accepts. The slope must move in `dvc metrics diff`. Pick your own row and value in place of these.

```bash
python -c "import pandas as pd; d = pd.read_csv('data/penguins.csv'); d.loc[0, 'body_mass_g'] += 500; d.to_csv('data/penguins.csv', index=False)"
dvc repro
dvc metrics diff | tee negative-control.txt
```

Now roll the data back and commit the record.

```bash
git checkout -- data/penguins.csv.dvc dvc.lock metrics.json
dvc checkout
dvc status
git add negative-control.txt
git commit --quiet -m "Record the negative control"
```

## Check everything

This is the full check, for steps 1 to 3. It clones your last commit and pulls the data, the way a colleague would. Then it plants bad deliveries, changes the policy, and watches what your pipeline does. It works in a throwaway clone, so your project is safe. It takes under a minute.

```bash
python ../check.py
```

With the bad delivery in place and no new checks, it fails like this:

In [6]:
# Recorded output. You don't need to run this.
run("python ../check.py")

$ python ../check.py
Note: you have uncommitted changes. This checks your last commit only.

PASS  The pipeline is committed
PASS  A fresh clone pulls everything into an empty cache
FAIL  dvc metrics diff in project/ isn't empty, so your workspace doesn't match your last commit. Roll back with the commands in step 1, or commit the result you want to keep.
FAIL  Your last commit doesn't hold the new delivery, cleaned. Finish step 2 and commit it, then dvc push.
PASS  Good data passes validate
FAIL  I planted a delivery with flipper lengths in inches, with the repairs off, and dvc repro did not stop at validate. Add a check to validate.py.
FAIL  I planted a delivery with repeated rows, with the repairs off, and dvc repro did not stop at validate. Add a check to validate.py.
FAIL  I planted a delivery with a misspelled species, with the repairs off, and dvc repro did not stop at validate. Add a check to validate.py.
FAIL  I changed drop_missing_sex and the report didn't rebuild. Declare t

## What you've got, and what you don't

A matching slope means the pipeline repeats. Whether flipper length really predicts mass this way is a question for you and your colleagues. The tools get you the first, and make the second possible.

## On your own project

Pick the dataset your last result came from. `dvc add` it, write down one thing that would make you refuse a delivery, and put that in a validate stage. Then change one row and see if your result moves. A local folder works as the remote until you have a bucket.

## About the data

Palmer penguins, 15 KB, 344 penguins, CC0 1.0. It's unchanged from the [palmerpenguins](https://allisonhorst.github.io/palmerpenguins/) package. `scripts/new_season.py` plants the inches and repeats, and those errors are not in the published data.

Cite it as: Horst AM, Hill AP, Gorman KB (2020). palmerpenguins: Palmer Archipelago (Antarctica) penguin data. R package version 0.1.0. doi:10.5281/zenodo.3960218. The data was first published in Gorman KB, Williams TD, Fraser WR (2014), PLoS ONE 9(3):e90081, doi:10.1371/journal.pone.0090081.